## Installation & Import

In [1]:
import os
import subprocess
import sys


def install_from_requirements(filepath="requirements.txt"):
    if not os.path.exists(filepath):
        print("Missing requirements.txt")
        return

    with open(filepath, "r", encoding="utf-8") as file:
        for line in file:
            package = line.strip()
            if not package or package.startswith("#"):
                continue
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", package])
                print(f"Sucessfully installed: {package}")
            except subprocess.CalledProcessError:
                print(f"Fail with lib: {package}")


install_from_requirements("requirements.txt")

Sucessfully installed: torch
Sucessfully installed: torchvision
Sucessfully installed: opencv-python
Sucessfully installed: Pillow
Sucessfully installed: numpy


In [7]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms.v2 as transforms
import cv2
import numpy as np
from torch.utils.data import Dataset

## Model initialization

In [4]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

COCO_INSTANCE_CATEGORY_NAMES = weights.meta["categories"]
COCO_INSTANCE_CATEGORY_NAMES[18] = 'Jerry' #separate category for my dog
cnn_model = fasterrcnn_resnet50_fpn(weights=weights)

## Custom dataset

In [11]:
num_classes = 91

#cnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, targets, transforms=None):
        self.image_paths = image_paths
        self.targets = targets
        self.transforms = transforms

    def __getitem__(self, idx):
        img = Image.open(r"pics_train\\"+self.image_paths[idx]).convert("RGB")
        img = transforms.functional.to_image(img)
        img = transforms.functional.to_dtype(img, scale=True)
        target = self.targets[idx]
        
        if self.transforms is not None:
            img = self.transforms(img)
        return img, target

    def __len__(self):
        return len(self.image_paths)

image_paths = ["photo_2026-06-01_18-16-30.jpg", "photo_2026-06-01_18-16-19.jpg", "photo_2026-06-01_18-15-47.jpg", "barbos1.jpg", "barbos2.jpg", "barbos3.jpg", "barbos4.jpg", "barbos5.jpg", "barbos6.jpg", "barbos7.jpg", "barbos8.jpg"]

tgt = [
    { 'boxes' : torch.tensor([[485, 97, 606, 186]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[274, 184, 355, 339]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[395, 485, 592, 772]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[884, 356, 1185, 644]]),  
      'labels' : torch.tensor([18])
    }
      ,
    { 'boxes' : torch.tensor([[935, 384, 1085, 682]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[748, 410, 1156, 636]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[728, 411, 1137, 683]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[686, 424, 1086, 811]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[901, 433, 1093, 684]]),  
      'labels' : torch.tensor([18])
    }
    ,
    { 'boxes' : torch.tensor([[940, 443, 1113, 674]]),  
      'labels' : torch.tensor([18])
    },
    { 'boxes' : torch.tensor([[828, 306, 1136, 560]]),  
      'labels' : torch.tensor([18])
    }
]

dog_fine = CustomDataset(image_paths, tgt)


## Training

In [12]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
cnn_model.to(device)

params = [p for p in cnn_model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

def collate_fn(batch):
    return tuple(zip(*batch))

cnn_model.train()

data_loader = torch.utils.data.DataLoader(dog_fine, batch_size=2, shuffle=True, collate_fn=collate_fn)

for epoch in range(100):
    for images, targets in data_loader:
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = cnn_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
    print(f"Epoch {epoch} loss: {losses.item()}")


Epoch 0 loss: 0.2958378493785858


KeyboardInterrupt: 

## Testing

In [ ]:
image_path = r"vids_test\\"
image_path += r"photo_2026-06-01_18-15-15.jpg" 
img = Image.open(image_path).convert("RGB")

img_tensor = F.to_tensor(img).unsqueeze(0)

cnn_model.eval()
with torch.no_grad():
    predictions = cnn_model(img_tensor)

boxes = predictions[0]['boxes']     
labels = predictions[0]['labels']    
scores = predictions[0]['scores']    

draw = ImageDraw.Draw(img)
threshold = .8 
for box, label, score in zip(boxes, labels, scores):
    class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
    if score > threshold and class_name=='Jerry':
        xmin, ymin, xmax, ymax = box.tolist()
        draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
        draw.text((xmin, ymin - 30), f"{class_name}: {score*100:.0f} %", fill="red", size = 40, font=ImageFont.truetype("arial.ttf", size=28))
img.show()

## Video

In [ ]:
test_video_path = r"vids_test\\"
test_video_path += "jerry_2_1.mp4"

detected_video_path = r"vids_detected\\"
detected_video_path += "jerry_2_1_detected.mp4"

cnn_model.rpn.nms_thresh = 0.1
cnn_model.roi_heads.nms_thresh = 0.01
cnn_model.eval()
def video_to_frames(video_path, frame_rate):

    cap = cv2.VideoCapture(video_path)
    
    frame_count = 0
    saved_count = 0
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    while cap.isOpened():
        
        ret, frame = cap.read()
        if not ret:
            break  
        if frame_count % frame_rate == 0:
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(rgb_image)
            img_tensor = F.to_tensor(pil_image).unsqueeze(0)

            with torch.no_grad():
                predictions = cnn_model(img_tensor)

            boxes = predictions[0]['boxes']     
            labels = predictions[0]['labels']    
            scores = predictions[0]['scores']    

            draw = ImageDraw.Draw(pil_image)
            threshold = .7
            for box, label, score in zip(boxes, labels, scores):
                class_name = COCO_INSTANCE_CATEGORY_NAMES[label]
                if score > threshold and class_name=='Jerry':
                    xmin, ymin, xmax, ymax = box.tolist()
                    draw.rectangle([(xmin, ymin), (xmax, ymax)], outline="red", width=3)
                    draw.text((xmin, ymin - 30), f"{'Jerry'}: {score*100:.0f} %", fill="red", size = 40, font=ImageFont.truetype("arial.ttf", size=28))
            
            rgb_image = cv2.cvtColor(np.array(pil_image), cv2.COLOR_BGR2RGB)
            height, width, layers = rgb_image.shape
            if frame_count == 0:
                video = cv2.VideoWriter(test_video_path, fourcc, 30, (width, height), isColor=True)
            video.write(rgb_image)
    
            saved_count += 1
            print('Frame', saved_count*frame_rate)
            
        frame_count += 1

    cap.release()
    video.release()
    
    print(f"Frames: {saved_count} out of {frame_count}")


video_to_frames(test_video_path, frame_rate=1)
